In [1]:
import scanpy as sc
import treedata as td
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(1, '/project/imoskowitz/yubin/SmoNull_NMPs_mesoderm_biased_analysis')
from src.III_celltype_annotation.run_celltype_annotation import run_find_markers, run_celltypist_annotation
from src.I_preprocessing.plot_preprocessing import plot_UMAP_custom


/project/imoskowitz/yubin/envs/Lineage_Tracing/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [2]:
sys.path.append("/project/imoskowitz/shared/sequencing.processed/Smo_null_snRNAseq2025/3-Brain_system/") 
from src.III_celltype_annotation.annotate_celltypist import (
    train_celltypist_model
)


#### Loading in Training data for the time point  of interest


In [3]:
data_dir = "output_data"
plot_dir = "output_plot"
base_path = "/project/imoskowitz/yubin/Lineage_Tree_Construction/"
output_path_data = base_path+data_dir+"/Celltypist/Predictions/"
output_path_plot = base_path+plot_dir+"/celltypist_annotation/"
adata_fname = "Processed_data/E8_5.h5td"

In [4]:
robin_training_data = sc.read('/project/imoskowitz/kdreyer/lab_datasets/002_Cardio_mesodermal_atlas/Cardio-mesodermal_atlas_formatted_E8.h5ad')

In [5]:
robin_training_data

AnnData object with n_obs × n_vars = 2847 × 29205
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'S.Score', 'G2M.Score', 'Phase', 'nCount_SCT', 'nFeature_SCT', 'Batch', 'Dataset', 'Old.groups', 'seurat_clusters', 'TimeStampMerged', 'Annotation', 'FHF_Pseudotime', 'JCF_Pseudotime', 'aSHF_Pseudotime', 'pSHF_Pseudotime', 'cell_type'
    var: 'features'
    uns: 'log1p'
    obsm: 'X_umap'
    layers: 'norm_counts', 'raw_counts'

In [7]:
list(robin_training_data.obs['cell_type'].unique())

['Pharyngeal mesoderm',
 'Mesenchyme',
 'Cardiomyocytes',
 'Paraxial mesoderm',
 'Nascent mesoderm',
 'Primitive streak',
 'Mixed mesoderm',
 'ExE mesoderm']

Mouse gastr data for background cells (ie not cardiac related) for annotating whole embryo

In [11]:
mouse_gastr_adata = sc.read("/project/imoskowitz/kdreyer/celltypist_models/source_data/gastrulation_extended_E75_E775_E80_E825_E85_E875.h5ad")

Subsetting to time point

In [15]:
mouse_gastr_E8_0 = mouse_gastr_adata[mouse_gastr_adata.obs['stage'] == "E8.0"]


In [17]:
mouse_gastr_E8_0.obs

,cell,sample,embryo_version,stage,somite_count,anatomy,S_score,G2M_score,phase,louvain,leiden,celltype_PijuanSala2019,celltype_extended_atlas,n_genes
26365,cell_30635,16,Original,E8.0,Pooled,Pooled,0.637718,0.239400,S,3,38,Caudal neurectoderm,Ectoderm,2618
26366,cell_30636,16,Original,E8.0,Pooled,Pooled,0.487921,0.306975,S,1,39,Paraxial mesoderm,Cardiopharyngeal progenitors FHF,2528
26367,cell_30638,16,Original,E8.0,Pooled,Pooled,0.024319,0.362608,G2M,5,17,Surface ectoderm,Pharyngeal endoderm,2540
26368,cell_30639,16,Original,E8.0,Pooled,Pooled,0.404758,0.566912,G2M,4,25,Caudal epiblast,Caudal epiblast,3718
26369,cell_30642,16,Original,E8.0,Pooled,Pooled,0.358236,0.406860,G2M,11,39,Pharyngeal mesoderm,Cardiopharyngeal progenitors FHF,3492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109469,cell_130401,35,Original,E8.0,Pooled,Pooled,0.475940,-0.111212,S,10,15,Intermediate mesoderm,Intermediate mesoderm,3096
109470,cell_130402,35,Original,E8.0,Pooled,Pooled,0.540834,0.308779,S,3,38,Caudal neurectoderm,Caudal epiblast,3743
109471,cell_130403,35,Original,E8.0,Pooled,Pooled,0.274621,0.327944,G2M,2,29,Mesenchyme,Mesenchyme,3401
109472,cell_130404,35,Original,E8.0,Pooled,Pooled,0.667279,-0.041098,S,10,15,Intermediate mesoderm,Intermediate mesoderm,2386


In [18]:
import gc
del mouse_gastr_adata
gc.collect()

270

#### Training Model

In [28]:
output_path_model = "/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/Celltypist/Models"
celltype_label = "Annotation"

# Name and file location of the reference adata
atlas_subset_fname = f"Cardio-mesodermal_atlas_formatted_E85.h5ad"
atlas_subset = sc.read_h5ad(
    "/project/imoskowitz/kdreyer/lab_datasets/002_Cardio_mesodermal_atlas/Cardio-mesodermal_atlas_formatted_E85.h5ad"
)

model, model_fname = train_celltypist_model(
    adata_atlas=atlas_subset, adata_atlas_fname=atlas_subset_fname,
    celltype_label=celltype_label, output_path_model=output_path_model,
    top_genes=1000
)

print(model_fname)

🍳 Preparing data before training
✂️ 8659 non-expressed genes are filtered out
🔬 Input data has 3170 cells and 20546 genes
⚖️ Scaling input data
🏋️ Training data using SGD logistic regression
🔎 Selecting features
🧬 7962 features are selected
🏋️ Starting the second round of training
🏋️ Training data using logistic regression
✅ Model training done!


Cardio-mesodermal_atlas_formatted_E85_model_top1000.pkl


#### Prediction

In [84]:
adata = td.read_h5td("/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/Processed_data/E8_5.h5td")
adata.X = adata.layers["Raw_count"]

AttributeError: 'dict' object has no attribute 'shape'

In [58]:
adata

TreeData object with n_obs × n_vars = 89230 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree', 'leiden_0.25', 'leiden_0.5', 'leiden_1.0', 'leiden_2.0'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'leiden_0.25', 'leiden_0.25_colors', 'leiden_0.5', 'leiden_0.5_colors', 'leiden_1.0', 'leiden_1.0_colors', 'leiden_2.0', 'leiden_2.0_colors', 'leiden_names', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scvi', 'X_umap', 'characters'
    varm: 'PCs'
    layers: 'Norm_count', 'Raw_count'
    obsp: 'connectivities', 'distances'
    obst: 'E8.5-R3-C1', 'E8.5-R2-C1', 'E8.5-R2-C2', 'E8.5-R1-C1', 'E8.5-R1-C3', 'E8.5-R1-C2', 'E8.5-R3-C3', 'E8.5-R3-C2', 'E8.5-R1-C4'

In [59]:
adata.layers['raw_counts'] = adata.layers['Raw_count'] # Changing due to compatibility issue with the model
del adata.layers['Raw_count']

In [60]:
output_path_model = "/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/Celltypist/Models/"
model_fname = "Cardio-mesodermal_atlas_formatted_E85_model_top1000.pkl"

In [61]:
### set clusters to use to get mode cell type per cluster
### make sure these match clustering resolutions/names in the adata
cluster_col_list = ["leiden_0.25", "leiden_0.5", "leiden_1.0", "leiden_2.0"]
clustering_name_list = ["l025", "l05", "l10", "l20"]

Note that For E8.5 the cell_type is instead called "Annotation"

In [62]:
adata_anno, top_celltypes_dict = run_celltypist_annotation(
    adata=adata, adata_atlas_path=None, time_pt=None, celltype_label= "cell_type", umap_coords_obsm="X_umap",
    output_path_data=output_path_data, output_path_model=output_path_model, 
    output_path_plot=output_path_plot, cluster_col_list=cluster_col_list,
    clustering_name_list=clustering_name_list, model_fname=model_fname,
    majority_voting=True
)

🔬 Input data has 89230 cells and 48004 genes
🔗 Matching reference genes in the model
🧬 7572 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 20
🗳️ Majority voting the predictions
✅ Majority voting done!
... storing 'tree' as categorical
... storing 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l025' as categorical
... storing 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l05' as categorical
... storing 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l10' as categorical
... storing 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l20' as categorical


In [65]:
adata_anno

TreeData object with n_obs × n_vars = 89230 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree', 'leiden_0.25', 'leiden_0.5', 'leiden_1.0', 'leiden_2.0', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_low_score', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l025', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l05', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l10', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l20'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'leiden_0.25', 'leiden_0.25_colors', 'leiden_0.5', 'leiden_0.5_colors', 'leiden_1.0', 'leiden_1.0_colors', 'leiden_2.0', 'leiden_2.0_colors', 'leiden_names', 'log1p', 'neighbors', 'pca', 'umap', 'Cardio-mesodermal_atlas_formatted_E8

#### Plotting results

In [4]:
adata = sc.read("/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/Celltypist/Predictions/E8_5/adata_anno_Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv.h5ad")

In [102]:
adata

TreeData object with n_obs × n_vars = 89230 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'log1p', 'neighbors', 'pca', 'umap', 'embryo_colors', 'cell_type_colors'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scvi', 'X_umap', 'characters'
    varm: 'PCs'
    layers: 'Norm_count', 'Raw_count'
    obsp: 'connectivities', 'distances'
    obst: 'E8.5-R3-C3', 'E8.5-R1-C4', 'E8.5-R2-C2', 'E8.5-R3-C1', 'E8.5-R1-C2', 'E8.5-R3-C2', 'E8.5-R2-C1', 'E8.5-R1-C1', 'E8.5-R1-C3'

In [ ]:
plot_UMAP_custom(adata=adata,
        umap_coords_obsm="X_umap",
        color_by="Cardio-mesodermal_atlas_formatted_E85_model_top1000_low_score",
        palette = "tab20",
        fig_title="Individual cell celltypist Confidence Score",
        output_path_plot=output_path_plot, 
        output_fname="umap_celltypist_individual_confidence.svg",
)

#### Adding predicted celltype back to original tdata

In [40]:
tdata = td.read_h5td("/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/Processed_data/E8_0.h5td")

In [45]:
tdata.obs["Robin_celltypist_annotation"] = adata.obs["Cardio-mesodermal_atlas_formatted_E8_model_top1000_mv_label"]
tdata.uns["Robin_celltypist_annotation_colors"] = adata.uns['Cardio-mesodermal_atlas_formatted_E8_model_top1000_mv_label_colors']

In [20]:
tdata

TreeData object with n_obs × n_vars = 89230 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scvi', 'X_umap', 'characters'
    varm: 'PCs'
    layers: 'Norm_count', 'Raw_count'
    obsp: 'connectivities', 'distances'
    obst: 'E8.5-R3-C3', 'E8.5-R1-C4', 'E8.5-R2-C2', 'E8.5-R3-C1', 'E8.5-R1-C2', 'E8.5-R3-C2', 'E8.5-R2-C1', 'E8.5-R1-C1', 'E8.5-R1-C3'

#### Retrieving Accidentally deleted E8.5 (oops)

In [18]:
converted_adata = td.TreeData(adata) # Taking the umap from this prediction adata, appending the treedata information to remake the original processed tdata

In [7]:
tdata = td.read_h5td("/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/E8_5.h5td")

In [26]:
converted_adata

TreeData object with n_obs × n_vars = 89230 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree', 'leiden_0.25', 'leiden_0.5', 'leiden_1.0', 'leiden_2.0', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_low_score', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l025', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l05', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l10', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l20'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_colors', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l025_colors', 'Cardio-mesodermal_atlas_formatted_E85_model_top1000_mv_label_mode_l05_colors', 

In [25]:
for tree in list(tdata.obst):
    converted_adata.obst[tree] = tdata.obst[tree]

In [ ]:
# converted_adata.write('/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/E8_5.h5td')

... storing 'tree' as categorical
